In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC                              # Modelo 3
from sklearn.naive_bayes import GaussianNB              # Modelo 4
from xgboost import XGBClassifier                       # Modelo 5 (instalar: pip install xgboost)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("✅ Todas las librerías importadas correctamente.")

✅ Todas las librerías importadas correctamente.


In [8]:
# Cargar el dataset original
df_original = pd.read_csv('Neo_Met_plants.csv')
Med_Plants = df_original.copy()

print("Dataset original cargado.")
print(f"Forma original: {Med_Plants.shape}")
print("\n✅ Todas las librerías importadas correctamente.")

Dataset original cargado.
Forma original: (1341, 20)

✅ Todas las librerías importadas correctamente.


In [9]:
# 1. Filtrar solo las filas donde Cereal sea 'barley' o 'wheat' para el target
df_wheat_barley = Med_Plants[Med_Plants['Cereal'].isin(['barley', 'wheat'])].copy()

print(f"Filas después del filtro (solo barley y wheat): {df_wheat_barley.shape[0]}")

# 2. Verificar cuántos hay de cada uno
print("\nConteo por tipo de cereal:")
print(df_wheat_barley['Cereal'].value_counts())

# 3. Crear la columna 'Cereal_encoded' (0 = barley, 1 = wheat)
df_wheat_barley['Cereal_encoded'] = (df_wheat_barley['Cereal'] == 'wheat').astype(int)

# Verificar que la codificación está correcta
print("\nVerificación de codificación:")

Filas después del filtro (solo barley y wheat): 1147

Conteo por tipo de cereal:
Cereal
barley    586
wheat     561
Name: count, dtype: int64

Verificación de codificación:


In [10]:
# 1. Codificar 'Chronological_Period_clean' como números, crear Período y Cuenca
le_periodo = LabelEncoder()
df_wheat_barley['Periodo_encoded'] = le_periodo.fit_transform(df_wheat_barley['Chronological_Period_clean'])

# 2. Codificar 'Mediterranean_Basin' como números
le_cuenca = LabelEncoder()
df_wheat_barley['Cuenca_encoded'] = le_cuenca.fit_transform(df_wheat_barley['Mediterranean_Basin'])

print("Nuevas columnas creadas:")
print(f"  - Periodo_encoded (valores únicos: {df_wheat_barley['Periodo_encoded'].nunique()})")
print(f"  - Cuenca_encoded (valores únicos: {df_wheat_barley['Cuenca_encoded'].nunique()})")

Nuevas columnas creadas:
  - Periodo_encoded (valores únicos: 8)
  - Cuenca_encoded (valores únicos: 3)


In [11]:
# 1. Definir las columnas predictoras, definir X e y
feature_columns = [
    'IRMS_d13C_Collagen',
    'd15N_Collagen',
    'Latitude_N',
    'Longitude_E',
    'Periodo_encoded',
    'Cuenca_encoded'
]

# Verificar que todas las columnas existen
missing_cols = [col for col in feature_columns if col not in df_wheat_barley.columns]
if missing_cols:
    print(f"⚠️ ATENCIÓN: Estas columnas no existen: {missing_cols}")
else:
    print("✓ Todas las columnas predictoras están presentes.")

# 2. Definir X e y
X = df_wheat_barley[feature_columns]
y = df_wheat_barley['Cereal_encoded']

print(f"\nVariables predictoras (X): {X.shape}")
print(f"Variable objetivo (y): {y.shape}")
print(f"\nDistribución de clases en 'y':")

✓ Todas las columnas predictoras están presentes.

Variables predictoras (X): (1147, 6)
Variable objetivo (y): (1147,)

Distribución de clases en 'y':


In [13]:
# Dividir los datos (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Tamaño entrenamiento: {X_train.shape[0]} muestras")
print(f"Tamaño prueba: {X_test.shape[0]} muestras")

Tamaño entrenamiento: 917 muestras
Tamaño prueba: 230 muestras


In [14]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = SVC(kernel='rbf', random_state=42)
model.fit(X_train_scaled, y_train)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [15]:
# Predecir sobre el conjunto de prueba
y_pred = model.predict(X_test_scaled)

# Métricas de rendimiento
accuracy = accuracy_score(y_test, y_pred)
print(f"Exactitud (Accuracy): {accuracy:.4f}")

print("\n--- Informe de clasificación ---")
print(classification_report(y_test, y_pred, target_names=['cebada (0)', 'trigo (1)']))

print("\n--- Matriz de Confusión ---")
print(confusion_matrix(y_test, y_pred))

# Comparar primeras predicciones
print("\n--- Comparación (primeros 10 casos de prueba) ---")
comparison = pd.DataFrame({
    'Real': y_test.values[:10],
    'Predicción': y_pred[:10],
    'Correcta?': y_test.values[:10] == y_pred[:10]
})
print(comparison)

Exactitud (Accuracy): 0.7130

--- Informe de clasificación ---
              precision    recall  f1-score   support

  cebada (0)       0.71      0.75      0.73       118
   trigo (1)       0.72      0.68      0.70       112

    accuracy                           0.71       230
   macro avg       0.71      0.71      0.71       230
weighted avg       0.71      0.71      0.71       230


--- Matriz de Confusión ---
[[88 30]
 [36 76]]

--- Comparación (primeros 10 casos de prueba) ---
   Real  Predicción  Correcta?
0     0           0       True
1     1           0      False
2     0           0       True
3     1           1       True
4     0           0       True
5     0           0       True
6     1           0      False
7     0           0       True
8     1           0      False
9     0           0       True
